# Universal Worker

> FastAPI server that runs inside isolated capability environments

In [ ]:
#| default_exp core.worker

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import argparse
import asyncio
import contextvars
import dataclasses
import importlib
import json
import os
import sys
import threading
import time
from contextlib import asynccontextmanager
from datetime import datetime
from typing import Any, AsyncIterator, Dict, Generator

import psutil
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse

from cjm_substrate.core.errors import CapabilityCancelledError, map_bare_exception_to_job_error
from cjm_substrate.core.platform import terminate_self

import logging
# CR-14: worker-process logging goes to the diagnostics STORE (structured
# records, contextvars-stamped job identity) when the proxy injected the
# env contract (CJM_DIAGNOSTICS_DB / CJM_WORKER_SESSION_ID / CJM_LOG_LEVEL);
# standalone/dev imports fall back to the pre-CR-14 stdout basicConfig.
# This module is a process entrypoint (SG-39 ownership note) — host code
# never imports it.
from cjm_substrate.core.diagnostics_store import install_worker_diagnostics
install_worker_diagnostics()

from cjm_substrate.core.capability import derive_structural_surface
from cjm_substrate.core.journal_store import SubstrateEventType
from cjm_substrate.core.wire import (
    ACCOUNTS_HEADER, CallEnvelope, ENVELOPE_BODY_KEY, begin_account_capture,
    drain_accounts, record_account, set_call_envelope, wire_encode,
)

The Universal Worker is a FastAPI server that:

1. **Dynamically loads** a capability class specified via CLI arguments
2. **Exposes HTTP endpoints** for capability lifecycle and execution
3. **Monitors parent process** ("Suicide Pact") to prevent zombie workers
4. **Reports telemetry** for resource scheduling decisions

```
Host Process                    Worker Process (Isolated Env)
┌──────────────┐               ┌─────────────────────────────┐
│ RemoteProxy  │──HTTP/JSON───▶│ Universal Worker (FastAPI)  │
│              │               │   ├─ /health                │
│  --ppid ─────│───────────────│──▶│ ├─ /initialize          │
│              │               │   ├─ /execute               │
└──────────────┘               │   ├─ /execute_stream        │
                               │   └─ PID Watchdog (Thread)  │
                               └─────────────────────────────┘
```

## EnhancedJSONEncoder

Custom JSON encoder that handles dataclasses and other common Python types that need serialization for HTTP responses.

In [ ]:
#| export
class EnhancedJSONEncoder(json.JSONEncoder):
    """JSON encoder that handles dataclasses and other common types.
    
    SG-52: datetime support added so JobError.occurred_at serializes cleanly
    when emitted via the typed `_job_error` terminal chunk in /execute_stream.
    """
    
    def default(
        self,
        o: Any # Object to encode
    ) -> Any: # JSON-serializable representation
        """Convert non-serializable objects to serializable form."""
        if dataclasses.is_dataclass(o) and not isinstance(o, type):
            return dataclasses.asdict(o)
        if isinstance(o, datetime):
            return o.isoformat()
        return super().default(o)

In [ ]:
# Test EnhancedJSONEncoder
from dataclasses import dataclass

@dataclass
class SampleConfig:
    name: str
    value: int

cfg = SampleConfig(name="test", value=42)
result = json.dumps(cfg, cls=EnhancedJSONEncoder)
print(f"Encoded: {result}")
assert json.loads(result) == {"name": "test", "value": 42}

Encoded: {"name": "test", "value": 42}


## PID Watchdog

The "Suicide Pact" pattern: if the Host process dies, the Worker must terminate itself to prevent zombie processes consuming resources.

In [ ]:
#| export
def parent_monitor(
    ppid: int # Parent process ID to monitor
) -> None:
    """Monitor parent process and terminate self if parent dies.
    
    This implements the "Suicide Pact" pattern: if the Host process dies,
    the Worker must terminate itself to prevent zombie processes.
    """
    try:
        while True:
            parent = psutil.Process(ppid)
            if parent.status() == psutil.STATUS_ZOMBIE:
                raise psutil.NoSuchProcess(ppid)
            time.sleep(1)
    except (psutil.NoSuchProcess, ProcessLookupError):
        print(f"[Worker {os.getpid()}] Parent {ppid} gone. Shutting down.", file=sys.stderr)
        # Use cross-platform self-termination
        terminate_self()

The watchdog runs in a daemon thread, checking every second if the parent process is still alive. When the parent dies (or becomes a zombie), the worker terminates itself using `terminate_self()` which handles cross-platform differences (SIGTERM on Unix, `os._exit()` on Windows).

## Application Factory

Creates the FastAPI application with all endpoints for capability communication.

### Factory reshape (NB-2, stage 2)

`create_app` was a single 439-line closure (over the 8K-char cell-split hard
trigger). It is now a thin assembler over module-level helpers grouped by
concern (the NB-2 "internal-helpers" option): capability loading, lifespan,
and five endpoint-registration groups. Endpoint bodies are verbatim from the
closure (each `_register_*` re-creates the same closure over
`capability_instance`); the only behavioral change is the typed wire envelope
(`wire_encode`) at the two result-serialization sites in the task group.

In [ ]:
#| export
def _load_capability_instance(
    module_name: str, # Python module path (e.g., "my_capability.capability")
    class_name: str   # Capability class name (e.g., "WhisperCapability")
):                    # Instantiated capability object
    """Dynamically load + instantiate the capability class.

    Runs synchronously before app construction so a load failure terminates
    the worker process with exit code 1 (matches pre-lifespan behavior;
    loading must succeed for the worker to be useful at all).
    """
    try:
        module = importlib.import_module(module_name)
        capability_cls = getattr(module, class_name)
        return capability_cls()
    except Exception as e:
        print(f"FATAL: Failed to load {module_name}:{class_name} - {e}", file=sys.stderr)
        sys.exit(1)

In [ ]:
#| export
def _make_lifespan(
    capability_instance  # The loaded capability object (closure-captured for shutdown cleanup)
):                   # FastAPI lifespan async context manager
    """Build the FastAPI lifespan that invokes capability.cleanup() on shutdown."""
    @asynccontextmanager
    async def lifespan(app: FastAPI) -> AsyncIterator[None]:
        """FastAPI lifespan replacing the deprecated @app.on_event("shutdown") path.

        Migration 2026-05-28: FastAPI deprecated `on_event` decorators in favor
        of the lifespan context manager (the DeprecationWarning fired in worker
        logs post-Session-A publish). Behavior is preserved verbatim:

        - Startup half is empty: the capability is already instantiated above
          BEFORE the FastAPI app is constructed (load failure exits the
          process before lifespan runs). No async startup work is needed.
        - Shutdown half: invoke capability.cleanup() before uvicorn exits.

        Session A 2026-05-27 rationale (still applies): both the parent_monitor
        watchdog (SIGTERM via terminate_self) and an external SIGTERM (e.g.,
        substrate's proxy stall-detection killing a wedged prefetch) route
        through uvicorn's graceful-shutdown path, which fires this lifespan
        teardown. Pre-Session-A the capability's cleanup() hook was only invoked
        on the manager-driven unload_capability path — never on watchdog or
        stall-triggered termination — so capabilities that spawn grandchild
        subprocesses (Voxtral-vLLM's vLLM server is the driving case) leaked
        them as orphans whenever the worker died abnormally. Cleanup failures
        are swallowed-with-log because they must NOT prevent uvicorn shutdown.

        cleanup() is synchronous (ToolCapability contract). Calling a sync
        function from this async block briefly blocks the event loop during
        shutdown, which is acceptable — uvicorn is already winding down.
        """
        # Startup: nothing to do (capability loaded above synchronously).
        yield
        # Shutdown: invoke capability.cleanup() best-effort.
        try:
            cleanup = getattr(capability_instance, "cleanup", None)
            if cleanup is not None:
                cleanup()
        except Exception as e:
            try:
                import logging as _lg
                _lg.getLogger(__name__).warning(
                    f"capability.cleanup() raised during worker shutdown: {e}"
                )
            except Exception:
                pass

    return lifespan

In [ ]:
#| export
def _register_identity_endpoints(
    app,             # FastAPI app under construction
    capability_instance, # The loaded capability object
) -> None:
    """/health + /stats: worker identity + process-subtree telemetry."""
    @app.get("/health")
    def health_check() -> Dict[str, Any]:
        """Health check endpoint."""
        return {
            "status": "running",
            "pid": os.getpid(),
            "name": getattr(capability_instance, "name", "unknown"),
            "version": getattr(capability_instance, "version", "unknown")
        }

    # Closure-captured Process instance + per-child baseline cache (SG-40 pattern).
    # psutil.Process.cpu_percent() caches the prior CPU-time baseline ON THE
    # INSTANCE: separate Process() instances pointing at the same PID DON'T
    # share the baseline, so a one-shot prime at module-main-block can't help
    # a /stats handler that creates a fresh Process() each call. Keeping the
    # Process instance alive in this closure means the SECOND /stats call onward
    # returns real CPU% for the worker (the first prime-call happens below).
    # Children get the same treatment via _child_procs (keyed by PID; new children
    # are primed on first appearance, return 0% that call, real% thereafter).
    _worker_proc = psutil.Process()
    _worker_proc.cpu_percent()  # Prime worker baseline (next call returns delta).
    _child_procs: Dict[int, psutil.Process] = {}

    @app.get("/stats")
    def stats() -> Dict[str, Any]:
        """Return process-tree resource usage for the worker subprocess.

        Aggregates RSS + CPU% across the worker AND its descendants in one
        psutil.children() walk so subprocess-spawning capabilities (Voxtral-vLLM's
        managed vLLM server is the driving case) report subtree totals rather
        than worker-only values. `subtree_pids` is emitted alongside so the
        substrate's GPU-attribution helper can intersect with the system-
        monitor capability's per-PID GPU enumeration without needing its own
        process-tree walk (which would duplicate this one).
        """
        worker_pid = os.getpid()

        # Single walk: descendants used for RSS + CPU% sums + subtree_pids list.
        try:
            descendants = _worker_proc.children(recursive=True)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            descendants = []  # No descendants → worker-only totals below.

        total_rss = _worker_proc.memory_info().rss
        total_cpu = _worker_proc.cpu_percent()  # Delta against the prime call in the closure.
        subtree_pids = [worker_pid]
        live_child_pids = set()
        for child in descendants:
            try:
                total_rss += child.memory_info().rss
                # Reuse the cached Process for delta-correctness (first call per child returns 0,
                # subsequent calls reflect actual CPU%). psutil.Process is hashable on (pid, create_time)
                # internally; we key by pid since descendants live shorter than the worker and pid recycling
                # within a single worker lifetime is extremely rare.
                cached = _child_procs.get(child.pid)
                if cached is None:
                    cached = child
                    cached.cpu_percent()  # Prime; this call returns 0.0
                    _child_procs[child.pid] = cached
                    total_cpu += 0.0
                else:
                    total_cpu += cached.cpu_percent()
                subtree_pids.append(child.pid)
                live_child_pids.add(child.pid)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                # Process died mid-walk OR we lack permission (rare for descendants); skip.
                pass

        # Evict cache entries for children that have exited (avoid unbounded growth
        # in long-running capabilities that frequently spawn-and-exit subprocesses).
        for stale_pid in list(_child_procs.keys()):
            if stale_pid not in live_child_pids:
                _child_procs.pop(stale_pid, None)

        return {
            "pid": worker_pid,
            "cpu_percent": total_cpu,  # Worker + descendants (subtree sum)
            "memory_rss_mb": total_rss / 1024 / 1024,
            "subtree_pids": subtree_pids,  # Worker + descendants; substrate intersects with sysmon GPU PIDs
            # SG-54: unit-agnostic measured usage the capability reported via
            # report_usage() during the last execute (None if it reported none).
            "usage": getattr(capability_instance, "_last_api_usage", None),
        }

    @app.get("/structural_surface")
    def structural_surface() -> Dict[str, Any]:
        """Pass-2 Thread 3 live companion: re-derive the structural surface
        at runtime so the manager can compare it against the manifest's
        stored witness hash and flag `surface_drift` (the CR-8 idiom's
        third instance). Same derivation the introspection script records
        at install/regenerate time.
        """
        return derive_structural_surface(type(capability_instance))



In [ ]:
#| export
def _register_lifecycle_endpoints(
    app,             # FastAPI app under construction
    capability_instance, # The loaded capability object
) -> None:
    """/initialize /prefetch /reconfigure /on_disable /on_enable /cleanup:
    the tool-capability lifecycle surface."""
    @app.post("/initialize")
    async def initialize(request: Request) -> Dict[str, str]:
        """Initialize or reconfigure the capability."""
        try:
            config = await request.json()
            capability_instance.initialize(config)
            return {"status": "ok"}
        except Exception as e:
            raise HTTPException(status_code=400, detail=str(e))

    @app.post("/prefetch")
    def prefetch() -> Dict[str, str]:
        """CR-4: invoke the capability's prefetch() hook for eager resource acquisition.
        
        Default ToolCapability.prefetch() is a no-op (SG-19 hook with opt-in
        semantics). Capabilities override when downstream callers benefit from
        eager model-download / cache-warming. Errors surface as 500 with the
        exception detail; idempotent calls are the capability's responsibility.
        """
        if not hasattr(capability_instance, "prefetch"):
            return {"status": "not_supported"}
        try:
            capability_instance.prefetch()
            return {"status": "prefetched"}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    @app.post("/reconfigure")
    async def reconfigure(request: Request) -> Dict[str, str]:
        """CR-4: invoke the capability's reconfigure(old_config, new_config) hook.
        
        Request body: `{"old_config": {...}, "new_config": {...}}`. Default
        ToolCapability.reconfigure() delegates to reconfigure_with_triggers
        which walks RELOAD_TRIGGER metadata on the capability's config_class.
        Capabilities predating CR-4 (no config_class) land in the silent-no-op
        branch — substrate falls back to /initialize when reconfigure is a
        no-op.
        """
        if not hasattr(capability_instance, "reconfigure"):
            return {"status": "not_supported"}
        try:
            data = await request.json()
            old_config = data.get("old_config")
            new_config = data.get("new_config")
            capability_instance.reconfigure(old_config, new_config)
            return {"status": "reconfigured"}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

    @app.post("/on_disable")
    def on_disable() -> Dict[str, str]:
        """CR-2: forward the substrate's on_disable signal to the loaded capability.
        
        Worker stays alive; capability's on_disable() hook gets a chance to
        release heavy resources (GPU memory, model files, etc.). Default
        ToolCapability.on_disable() is a no-op so capabilities that don't
        opt in see no behavior change.
        """
        if hasattr(capability_instance, 'on_disable'):
            try:
                capability_instance.on_disable()
                return {"status": "disabled"}
            except Exception as e:
                return {"status": "error", "detail": str(e)}
        return {"status": "not_supported"}

    @app.post("/on_enable")
    def on_enable() -> Dict[str, str]:
        """CR-2: forward the substrate's on_enable signal to the loaded capability.
        
        Capability can eagerly re-acquire heavy resources or rely on lazy
        re-acquisition via the next execute() call. Default
        ToolCapability.on_enable() is a no-op.
        """
        if hasattr(capability_instance, 'on_enable'):
            try:
                capability_instance.on_enable()
                return {"status": "enabled"}
            except Exception as e:
                return {"status": "error", "detail": str(e)}
        return {"status": "not_supported"}

    @app.post("/cleanup")
    def cleanup() -> Dict[str, str]:
        """Clean up capability resources."""
        if hasattr(capability_instance, 'cleanup'):
            capability_instance.cleanup()
        return {"status": "cleaned"}



In [ ]:
#| export
def _register_config_endpoints(
    app,             # FastAPI app under construction
    capability_instance, # The loaded capability object
) -> None:
    """/config_schema /config /config_options: the config surface."""
    @app.get("/config_schema")
    def get_config_schema() -> Dict[str, Any]:
        """Return JSON Schema for capability configuration."""
        if hasattr(capability_instance, 'get_config_schema'):
            return capability_instance.get_config_schema()
        return {}

    @app.get("/config")
    def get_config() -> Dict[str, Any]:
        """Return current capability configuration."""
        if hasattr(capability_instance, 'get_current_config'):
            cfg = capability_instance.get_current_config()
            # Ensure dataclasses are serialized
            return json.loads(json.dumps(cfg, cls=EnhancedJSONEncoder))
        return {}

    @app.get("/config_options")
    def get_config_options() -> Dict[str, Any]:
        """CR-11: return live config option domains (dynamic schema enrichment)."""
        if hasattr(capability_instance, 'get_config_options'):
            opts = capability_instance.get_config_options()
            # FieldOptions/ConfigOption dataclasses -> JSON-serializable dicts
            return json.loads(json.dumps(opts, cls=EnhancedJSONEncoder))
        return {}



In [ ]:
#| export
def _load_adapters(
    capability_instance,  # The loaded tool-capability instance
    adapter_specs,    # List of "module:ClassName" impl specs (host-matched)
) -> Dict[str, Any]:  # task_name -> bound adapter instance
    """Instantiate task-adapter impls bound to this worker's tool (CR-17 pt 2).

    Each spec was matched HOST-side (adapter-manifest protocol members vs the
    capability's recorded structural surface) before reaching the worker, so a
    spec failing HERE is an INSTALL gap (interface lib missing from this env),
    not a compatibility miss — log loudly, skip, keep serving /execute.
    Binding convention: `AdapterClass(capability_instance)`; keyed by the class's
    `task_name` ClassVar.
    """
    adapters: Dict[str, Any] = {}
    for spec in adapter_specs or []:
        try:
            module_name, class_name = spec.rsplit(":", 1)
            mod = importlib.import_module(module_name)
            cls = getattr(mod, class_name)
            task_name = getattr(cls, "task_name", "") or ""
            if not task_name:
                logging.error(f"Adapter {spec!r} has no task_name; skipping")
                continue
            if task_name in adapters:
                logging.error(
                    f"Task {task_name!r} already bound to "
                    f"{type(adapters[task_name]).__name__}; skipping {spec!r}")
                continue
            adapters[task_name] = cls(capability_instance)
            logging.info(f"Bound adapter {class_name} for task {task_name!r}")
        except Exception as e:
            logging.error(f"Failed to load adapter {spec!r}: {e}")
    return adapters


In [ ]:
#| export
def _apply_call_envelope(
    data: Dict[str, Any],  # The decoded request body
) -> None:
    """CR-14: decode the wire envelope into the worker-side contextvar.

    Set WITHOUT reset: ASGI handles each request in its own asyncio task, so
    the context (and the var) dies with the request — and for
    /execute_stream the response iteration runs in the SAME request task
    AFTER the endpoint returns, which is exactly why a reset-before-return
    would lose the identity (Starlette's iterate_in_threadpool copies the
    request task's context per chunk). An absent envelope leaves the var
    None — records stay honestly unattributed.

    CR-14 follow-up: also begins a fresh account-capture span (same no-reset
    semantics) so in-worker substrate-family accounts (TASK_ACCOUNT /
    RESULT_SAVED / CACHE_HIT) accumulate per call and ride back on the
    response header — see `_accounts_headers`.
    """
    env_wire = data.get(ENVELOPE_BODY_KEY)
    if env_wire:
        set_call_envelope(CallEnvelope.from_wire(env_wire))
    begin_account_capture()


def _accounts_headers() -> Dict[str, str]:
    """Drain the call span's recorded accounts into the response header dict.

    Empty dict when nothing was recorded (header absent — old hosts and
    account-less calls see byte-identical responses). ASCII JSON
    (`ensure_ascii` default) keeps the header latin-1-safe.
    """
    accounts = drain_accounts()
    if not accounts:
        return {}
    return {ACCOUNTS_HEADER: json.dumps(accounts, default=str)}


def _register_task_endpoints(
    app,             # FastAPI app under construction
    capability_instance, # The loaded capability object
    adapters=None,   # task_name -> bound adapter instance (CR-17 pt 2; stage 4)
) -> None:
    """/execute /execute_stream /cancel /progress /task: the task channel.

    Stage 2 (typed wire layer): both result-serialization sites pass
    through `wire_encode`, so results whose DTO classes are registered
    via `@wire_type` cross the boundary in the tagged envelope and arrive
    typed at the proxy; unregistered results serialize exactly as before.

    CR-14 (stage 7): each call decodes the per-call envelope into the
    contextvar and carries it into the executor thread via
    `contextvars.copy_context()` (run_in_executor does NOT copy context by
    itself) — the diagnostics handler stamps every capability log record with
    exact job identity, replacing the timestamp-window heuristic.

    CR-14 follow-up: the unary paths (/execute, /task) return recorded
    accounts on the `X-CJM-Accounts` response header (success AND the
    `_job_error` 500 — a failed call still reports the accounts recorded
    before the failure). The host journals them with `worker_reported=True`.
    """
    @app.post("/execute")
    async def execute(request: Request) -> Any:
        """Execute capability's main functionality.
        
        Runs in a thread pool so the event loop stays free to serve
        concurrent requests (e.g., /progress polling during long operations).
        
        CR-4: resets the capability's `_cancel_requested` flag before invoking
        execute() so cancellation doesn't leak from a previous job. After
        execute() returns or raises, leaves the flag in its post-call state
        (cancel() may have been called during cleanup; substrate decides what
        to do with that on next /cancel signal).
        
        CR-4: CapabilityCancelledError → HTTP 409 (operator-requested cancellation;
        distinct from 500's "real failure" so the proxy can surface a typed
        "cancelled" state to callers and JobQueue can render the job state
        accordingly per CR-6).
        """
        data = await request.json()
        args = data.get("args", [])
        kwargs = data.get("kwargs", {})
        _apply_call_envelope(data)  # CR-14: job identity + account span for this call
        
        # CR-4: reset cancellation flag before invoking execute() so stale
        # state from a previous job doesn't immediately raise.
        if hasattr(capability_instance, "_cancel_requested"):
            capability_instance._cancel_requested = False
        # SG-54: reset measured usage so a failed / usage-less call can't inherit
        # a prior run's usage when the substrate reads /stats post-execute.
        capability_instance._last_api_usage = None
        
        try:
            loop = asyncio.get_event_loop()
            # CR-14: copy_context AFTER the envelope set, so the executor
            # thread's capability code (and its logger calls) sees the identity.
            ctx = contextvars.copy_context()
            result = await loop.run_in_executor(
                None, lambda: ctx.run(capability_instance.execute, *args, **kwargs)
            )
            # Typed wire envelope for registered result DTOs (stage 2);
            # EnhancedJSONEncoder still flattens unregistered dataclasses
            # and datetimes exactly as before.
            json_str = json.dumps(wire_encode(result), cls=EnhancedJSONEncoder)
            return JSONResponse(content=json.loads(json_str),
                                headers=_accounts_headers())
        except CapabilityCancelledError as e:
            # CR-4: cooperative cancellation surfaces as 409 Conflict so the
            # proxy can distinguish "operator cancelled" from "real capability
            # failure" (500). The detail body carries the capability name for
            # debugging context.
            raise HTTPException(status_code=409, detail=str(e))
        except Exception as e:
            import traceback
            traceback.print_exc()
            # SG-52 parity for the UNARY path (stage-3 stress-test finding,
            # ledger G7): the 500 body carries the SAME `{"_job_error":
            # <JobError dict>}` sentinel shape as the stream's terminal
            # chunk, so the proxy raises the typed exception client-side
            # (CapabilityResourceError et al) and CR-7's reactive retry can
            # actually see resource errors from plain /execute calls. The
            # old bare-string detail collapsed every failure to RuntimeError
            # host-side, leaving the retry path blind on this channel.
            job_error = map_bare_exception_to_job_error(
                e, capability_name=getattr(capability_instance, "name", "unknown"),
            )
            body = json.loads(json.dumps({"_job_error": job_error},
                                         cls=EnhancedJSONEncoder))
            return JSONResponse(status_code=500, content=body,
                                headers=_accounts_headers())

    _adapters = adapters or {}

    @app.post("/task")
    async def task_call(request: Request) -> Any:
        """Typed task-adapter dispatch (CR-17 pt 2; stage 4).

        Body: {"task": <task_name>, "method": <adapter method>, "kwargs": {...}}.
        Kwargs-only by design — typed task methods have named parameters; the
        explicit task channel never overloads `action=` (that stays the tool's
        native in-worker dispatch). Results pass through `wire_encode` (typed
        DTOs cross the boundary in the tagged envelope); failures carry the
        same `{"_job_error": ...}` 500 body as /execute (G7 typed unary error
        channel) so CR-7 retry sees typed categories from day one.

        CR-14 follow-up: the worker itself records a TASK_ACCOUNT for every
        adapter dispatch (zero capability cooperation — derived at the substrate
        boundary): task, method, ok, duration. On failure the account carries
        ok=False + the JobError category, riding the same 500's header.
        """
        data = await request.json()
        task = data.get("task")
        method_name = data.get("method") or ""
        kwargs = data.get("kwargs", {})
        _apply_call_envelope(data)  # CR-14: job identity + account span for this call
        adapter = _adapters.get(task)
        if adapter is None:
            raise HTTPException(
                status_code=404,
                detail=f"No adapter bound for task {task!r} (bound: {sorted(_adapters)})")
        if not method_name or method_name.startswith("_"):
            raise HTTPException(
                status_code=404, detail=f"Invalid task method {method_name!r}")
        fn = getattr(adapter, method_name, None)
        if not callable(fn):
            raise HTTPException(
                status_code=404,
                detail=f"Task {task!r} adapter has no method {method_name!r}")

        # Cancel-flag + usage reset parity with /execute (the adapter drives
        # the same tool instance underneath).
        if hasattr(capability_instance, "_cancel_requested"):
            capability_instance._cancel_requested = False
        capability_instance._last_api_usage = None
        _t0 = time.monotonic()
        try:
            loop = asyncio.get_event_loop()
            ctx = contextvars.copy_context()  # CR-14: carry identity into the executor
            result = await loop.run_in_executor(None, lambda: ctx.run(lambda: fn(**kwargs)))
            json_str = json.dumps(wire_encode(result), cls=EnhancedJSONEncoder)
            record_account(SubstrateEventType.TASK_ACCOUNT.value, {
                "task": task, "method": method_name, "ok": True,
                "duration_s": round(time.monotonic() - _t0, 3),
            })
            return JSONResponse(content=json.loads(json_str),
                                headers=_accounts_headers())
        except CapabilityCancelledError as e:
            raise HTTPException(status_code=409, detail=str(e))
        except HTTPException:
            raise
        except Exception as e:
            import traceback
            traceback.print_exc()
            job_error = map_bare_exception_to_job_error(
                e, capability_name=getattr(capability_instance, "name", "unknown"),
            )
            record_account(SubstrateEventType.TASK_ACCOUNT.value, {
                "task": task, "method": method_name, "ok": False,
                "duration_s": round(time.monotonic() - _t0, 3),
                "error_category": job_error.category,
            })
            body = json.loads(json.dumps({"_job_error": job_error},
                                         cls=EnhancedJSONEncoder))
            return JSONResponse(status_code=500, content=body,
                                headers=_accounts_headers())

    @app.post("/execute_stream")
    async def execute_stream(request: Request) -> StreamingResponse:
        """Execute capability with streaming response (NDJSON).
        
        SG-51: resets `_cancel_requested` at the start of iter_response (parity
        with /execute) so a leftover cancel flag from a previous job doesn't
        cause the first check_cancel() to raise on the new stream.
        
        SG-52: typed error chunks via `{"_job_error": <JobError dict>}` terminal
        chunk replace the undocumented `{"error": str(e)}` shape. The wire-format
        contract: ANY exception raised during streaming is converted via
        `map_bare_exception_to_job_error` (CR-5 default classification) and
        emitted as the final NDJSON line with the `_job_error` sentinel key.
        Capability output chunks never carry that key, so consumers (proxy +
        downstream) can detect terminal errors with a single dict membership
        check. The proxy's execute_stream then raises a typed exception
        client-side, mirroring /execute's HTTP 409 → CapabilityCancelledError flow.

        CR-14: `_apply_call_envelope`'s no-reset semantics matter HERE — the
        response iteration runs in this request task after the endpoint
        returns, and each chunk's threadpool hop copies the task context,
        so the capability's in-stream logger calls stay job-stamped.

        CR-14 follow-up honest limit: accounts do NOT ride this path —
        response headers are sent before execution ends. A terminal sentinel
        chunk (the `_job_error` dialect) is the seam-admitted carrier if a
        streaming adopter ever needs them.
        """
        data = await request.json()
        args = data.get("args", [])
        kwargs = data.get("kwargs", {})
        _apply_call_envelope(data)  # CR-14: identity persists through iteration

        def iter_response() -> Generator[str, None, None]:
            # SG-51: reset cancel flag so stale state from a previous job doesn't
            # immediately raise inside the capability's execute_stream iterator.
            if hasattr(capability_instance, "_cancel_requested"):
                capability_instance._cancel_requested = False
            try:
                if hasattr(capability_instance, 'execute_stream'):
                    iterator = capability_instance.execute_stream(*args, **kwargs)
                else:
                    # Fallback: wrap single result
                    iterator = [capability_instance.execute(*args, **kwargs)]
                
                for chunk in iterator:
                    # Line-delimited JSON (NDJSON)
                    yield json.dumps(wire_encode(chunk), cls=EnhancedJSONEncoder) + "\n"
            except Exception as e:
                # SG-52: emit typed JobError as the terminal chunk. Proxy +
                # downstream consumers detect the `_job_error` sentinel key
                # and raise the corresponding typed exception client-side
                # (CapabilityCancelledError, CapabilityTransientError, etc.).
                job_error = map_bare_exception_to_job_error(
                    e, capability_name=getattr(capability_instance, "name", "unknown"),
                )
                yield json.dumps({"_job_error": job_error}, cls=EnhancedJSONEncoder) + "\n"

        return StreamingResponse(iter_response(), media_type="application/x-ndjson")

    @app.post("/cancel")
    def cancel() -> Dict[str, str]:
        """Cancel any running execution."""
        if hasattr(capability_instance, 'cancel'):
            try:
                capability_instance.cancel()
                return {"status": "cancelled"}
            except Exception as e:
                return {"status": "error", "detail": str(e)}
        return {"status": "not_supported"}

    @app.get("/progress")
    def get_progress() -> Dict[str, Any]:
        """Get current execution progress."""
        return {
            "progress": getattr(capability_instance, '_progress', 0.0),
            "message": getattr(capability_instance, '_status_message', "")
        }


In [ ]:
#| export
def _register_monitor_endpoints(
    app,             # FastAPI app under construction
    capability_instance, # The loaded capability object
    class_name: str, # Capability class name (for 404 detail messages)
) -> None:
    """/get_system_status /list_processes: CR-3 typed MonitorToolProtocol accessors."""
    @app.post("/get_system_status")
    def get_system_status_endpoint() -> Dict[str, Any]:
        """CR-3: typed MonitorToolProtocol accessor. Returns SystemStats.to_dict().
        
        Status code taxonomy (intentional, per CR-3 review):
        - 404: capability is not a monitor (configuration error — don't mask).
              Substrate's `system_monitor` was wired to the wrong capability.
        - 501: capability is a monitor but raised NotImplementedError from its
              get_system_status() default body (legacy monitor predating CR-3 that
              opted out). Proxy falls back to /execute("get_system_status").
        - 500: capability's get_system_status() raised some other exception. Real
              failure; do not silently fall back.
        - 200: typed call succeeded; body is SystemStats.to_dict().
        """
        if not hasattr(capability_instance, 'get_system_status'):
            raise HTTPException(
                status_code=404,
                detail=(
                    f"Capability {getattr(capability_instance, 'name', class_name)!r} "
                    f"is not a monitor capability (no get_system_status method)."
                ),
            )
        try:
            stats = capability_instance.get_system_status()
        except NotImplementedError as exc:
            raise HTTPException(status_code=501, detail=str(exc) or "get_system_status not implemented")
        # Serialize via EnhancedJSONEncoder so SystemStats dataclass flattens
        # to its plain dict (and tolerates dict returns from non-typed monitors).
        if hasattr(stats, 'to_dict'):
            return stats.to_dict()
        return json.loads(json.dumps(stats, cls=EnhancedJSONEncoder))

    @app.post("/list_processes")
    def list_processes_endpoint() -> Any:
        """CR-3: typed MonitorToolProtocol accessor. Returns list of ProcessStats.to_dict().
        
        Same 404/501/500/200 taxonomy as /get_system_status. MonitorToolProtocol's
        default list_processes() returns `[]`, so non-implementing monitors
        return 200 with an empty list rather than 501.
        """
        if not hasattr(capability_instance, 'list_processes'):
            raise HTTPException(
                status_code=404,
                detail=(
                    f"Capability {getattr(capability_instance, 'name', class_name)!r} "
                    f"is not a monitor capability (no list_processes method)."
                ),
            )
        try:
            procs = capability_instance.list_processes()
        except NotImplementedError as exc:
            raise HTTPException(status_code=501, detail=str(exc) or "list_processes not implemented")
        # Each ProcessStats has .to_dict(); EnhancedJSONEncoder handles either
        # dataclass instances or pre-converted dicts uniformly.
        return json.loads(json.dumps(procs, cls=EnhancedJSONEncoder))



In [ ]:
#| export
def create_app(
    module_name: str, # Python module path (e.g., "my_capability.capability")
    class_name: str,  # Capability class name (e.g., "WhisperCapability")
    adapter_specs=None # CR-17 pt 2: "module:ClassName" adapter impl specs to bind in-worker
) -> FastAPI: # Configured FastAPI application
    """Create FastAPI app that hosts the specified capability.

    NB-2 reshape (stage 2): a thin assembler — load the capability, build the
    lifespan, register the endpoint groups. Endpoint behavior lives in the
    module-level `_register_*` helpers above.
    """
    capability_instance = _load_capability_instance(module_name, class_name)
    adapters = _load_adapters(capability_instance, adapter_specs)
    app = FastAPI(title="Capability Worker", lifespan=_make_lifespan(capability_instance))
    _register_identity_endpoints(app, capability_instance)
    _register_lifecycle_endpoints(app, capability_instance)
    _register_config_endpoints(app, capability_instance)
    _register_task_endpoints(app, capability_instance, adapters)
    _register_monitor_endpoints(app, capability_instance, class_name)
    return app

### Endpoint Summary

| Endpoint | Method | Purpose |
|----------|--------|----------------------------------------------|
| `/health` | GET | Health check with PID and capability identity |
| `/stats` | GET | Process telemetry (CPU, memory) for scheduler |
| `/initialize` | POST | Configure/reconfigure capability |
| `/prefetch` | POST | CR-4: eager resource acquisition (SG-19 hook) |
| `/reconfigure` | POST | CR-4: hot-reload via reconfigure(old, new); declarative `RELOAD_TRIGGER` dispatch |
| `/config_schema` | GET | JSON Schema for UI generation |
| `/config` | GET | Current configuration values |
| `/execute` | POST | Execute capability, return JSON. CR-4: `CapabilityCancelledError` → 409 (operator cancellation) vs 500 (real failure) |
| `/execute_stream` | POST | Execute with streaming NDJSON response. SG-51: resets `_cancel_requested` at start of iteration. SG-52: errors emit as terminal `{"_job_error": <JobError dict>}` chunk with CR-5 typed classification (replaces undocumented `{"error": str(e)}` shape). |
| `/cancel` | POST | Request cancellation of running execution. CR-4: default `ToolCapability.cancel()` sets `_cancel_requested` flag + fires registered callbacks |
| `/progress` | GET | Get current execution progress and status |
| `/on_disable` | POST | CR-2: signal capability to release heavy resources (worker stays alive) |
| `/on_enable` | POST | CR-2: signal capability to re-acquire resources |
| `/get_system_status` | POST | CR-3: typed `MonitorToolProtocol.get_system_status()` — returns `SystemStats.to_dict()`; 404 if not a monitor, 501 if monitor raises `NotImplementedError` |
| `/list_processes` | POST | CR-3: typed `MonitorToolProtocol.list_processes()` — returns list of `ProcessStats.to_dict()`; same 404/501 taxonomy |
| `/cleanup` | POST | Release capability resources |

#### `/execute_stream` wire-format contract (SG-52)

Each NDJSON line is one of:

- **Capability output chunk**: a JSON object yielded by the capability's `execute_stream` (or wrapped single result from `execute`). Shape is capability-defined.
- **Terminal error chunk**: `{"_job_error": {<JobError dict>}}`. Emitted at most once and only as the last chunk. The `_job_error` sentinel key never appears in capability output chunks. The nested dict carries CR-5's full `JobError` structure: `category`, `message`, `retriable`, `original_exc_repr`, `traceback`, `retry_after_seconds`, `fields_invalid`, `resource_shortfall`, `capability_name`, `capability_instance_id`, `occurred_at` (ISO 8601 string).

`CapabilityCancelledError` is identifiable in the error chunk via `category == "transient"` combined with `original_exc_repr` prefix `"CapabilityCancelledError"` — the proxy uses this to raise the typed exception client-side rather than yielding the error chunk to downstream consumers.

In [ ]:
#| hide
# CR-3 follow-up: exercise the 404 / 501 status code distinction on
# /get_system_status + /list_processes via FastAPI's in-process TestClient.
# Without this test, the 404 and 501 branches in worker.py are pure defensive
# code — no production capability currently raises NotImplementedError from its
# typed methods, so the 501 branch never fires under normal operation. Locking
# the status-code contract here means the proxy's 501→/execute fallback and
# 404→ERROR-log paths have a worker-side counterpart that's verified to fire.
import sys as _sys_cr3
import types as _types_cr3

from fastapi.testclient import TestClient as _TestClient


# Fake capability module injected into sys.modules so create_app's dynamic-import
# path can find the stubs. Each stub picks a different failure mode to exercise.
_fake_mod = _types_cr3.ModuleType("_cr3_worker_test_module")


class _NotAMonitorCapability:
    """Stub capability that is NOT a monitor — lacks get_system_status / list_processes
    entirely. Worker should return 404 (configuration error: wrong capability type)."""
    name = "not-a-monitor"
    version = "1.0.0"
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs): return {"ok": True}
    def get_config_schema(self): return {}
    def get_current_config(self): return {}


class _RaisingMonitorCapability:
    """Stub capability that IS a monitor shape (has typed methods) but explicitly
    raises NotImplementedError — simulates a legacy monitor opting out of CR-3
    typed surface. Worker should return 501 (proxy then falls back to /execute)."""
    name = "raising-monitor"
    version = "1.0.0"
    def initialize(self, config=None): pass
    def execute(self, command="get_system_status", **kwargs):
        if command == "get_system_status":
            return {"gpu_type": "LEGACY", "cpu_percent": 12.0}
        return {}
    def get_config_schema(self): return {}
    def get_current_config(self): return {}
    def get_system_status(self):
        raise NotImplementedError("opted out of typed surface")
    def list_processes(self):
        raise NotImplementedError("opted out of typed surface")


_fake_mod._NotAMonitorCapability = _NotAMonitorCapability
_fake_mod._RaisingMonitorCapability = _RaisingMonitorCapability
_sys_cr3.modules["_cr3_worker_test_module"] = _fake_mod


def _cr3_worker_status_codes_check():
    # Case A: 404 — capability lacks the typed monitor methods entirely.
    # Loud configuration error: substrate's system_monitor wired wrong.
    app_404 = create_app("_cr3_worker_test_module", "_NotAMonitorCapability")
    with _TestClient(app_404) as client:
        resp = client.post("/get_system_status")
        assert resp.status_code == 404, f"expected 404, got {resp.status_code}: {resp.text}"
        detail = resp.json()["detail"]
        assert "not a monitor capability" in detail, f"missing intent in detail: {detail!r}"
        
        resp = client.post("/list_processes")
        assert resp.status_code == 404, f"expected 404, got {resp.status_code}: {resp.text}"
        assert "not a monitor capability" in resp.json()["detail"]
    
    # Case B: 501 — capability has the typed methods but raises NotImplementedError.
    # Proxy will fall back to /execute("get_system_status") returning the
    # legacy dict shape. Worker's job is just to surface 501 cleanly.
    app_501 = create_app("_cr3_worker_test_module", "_RaisingMonitorCapability")
    with _TestClient(app_501) as client:
        resp = client.post("/get_system_status")
        assert resp.status_code == 501, f"expected 501, got {resp.status_code}: {resp.text}"
        assert "opted out" in resp.json()["detail"]
        
        resp = client.post("/list_processes")
        assert resp.status_code == 501, f"expected 501, got {resp.status_code}: {resp.text}"
        
        # Sanity: /execute still works on the same capability (proxy's fallback path
        # depends on this — the worker keeps serving /execute even when typed
        # endpoints return 501).
        resp = client.post("/execute", json={"args": [], "kwargs": {"command": "get_system_status"}})
        assert resp.status_code == 200, f"/execute should work: {resp.status_code} {resp.text}"
        body = resp.json()
        assert body == {"gpu_type": "LEGACY", "cpu_percent": 12.0}


_cr3_worker_status_codes_check()
print("CR-3 worker 404/501 status codes: PASS")

In [ ]:
#| hide
# CR-4 worker endpoint tests: /prefetch + /reconfigure + execute() cancellation
# → 409 status. Same TestClient + sys.modules injection trick used in the CR-3
# tests above. Each stub capability instruments a specific behavior.

from cjm_substrate.core.errors import CapabilityCancelledError as _CapabilityCancelledError_CR4

_fake_mod_cr4 = _types_cr3.ModuleType("_cr4_worker_test_module")


class _CR4PrefetchCapability:
    """Capability tracking prefetch + reconfigure invocations for endpoint verification."""
    name = "cr4-prefetch-capability"
    version = "1.0.0"
    # Class-level so the test class can verify across instances; the worker
    # creates one instance via capability_cls() so a class-level counter is fine.
    prefetch_calls = 0
    reconfigure_calls = []  # list of (old, new) tuples
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs): return {"ok": True}
    def get_config_schema(self): return {}
    def get_current_config(self): return {}
    def prefetch(self):
        type(self).prefetch_calls += 1
    def reconfigure(self, old_config, new_config):
        type(self).reconfigure_calls.append((old_config, new_config))


class _CR4PrefetchRaisingCapability(_CR4PrefetchCapability):
    """Capability whose prefetch() raises — worker returns 500 with detail."""
    name = "cr4-prefetch-raising-capability"
    def prefetch(self):
        raise RuntimeError("prefetch boom")


class _CR4CancellingCapability:
    """Capability whose execute() raises CapabilityCancelledError to drive the 409 path."""
    name = "cr4-cancelling-capability"
    version = "1.0.0"
    _cancel_requested = False  # CR-4 flag (class-level default)
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs):
        # Simulate a capability that voluntarily surfaces cooperative cancellation.
        # In real use, capability code calls self.check_cancel() and it raises when
        # the substrate has flipped the flag via /cancel; this stub just raises
        # directly to keep the test independent of timing.
        raise _CapabilityCancelledError_CR4(self.name)
    def get_config_schema(self): return {}
    def get_current_config(self): return {}


class _CR4FailingCapability:
    """Capability whose execute() raises a real RuntimeError — worker returns 500.
    Asserts the 500/409 distinction (real failure vs cancellation)."""
    name = "cr4-failing-capability"
    version = "1.0.0"
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs):
        raise RuntimeError("simulated capability failure")
    def get_config_schema(self): return {}
    def get_current_config(self): return {}


_fake_mod_cr4._CR4PrefetchCapability = _CR4PrefetchCapability
_fake_mod_cr4._CR4PrefetchRaisingCapability = _CR4PrefetchRaisingCapability
_fake_mod_cr4._CR4CancellingCapability = _CR4CancellingCapability
_fake_mod_cr4._CR4FailingCapability = _CR4FailingCapability
_sys_cr3.modules["_cr4_worker_test_module"] = _fake_mod_cr4


def _cr4_worker_endpoints_check():
    # /prefetch success path
    _CR4PrefetchCapability.prefetch_calls = 0
    _CR4PrefetchCapability.reconfigure_calls = []
    app = create_app("_cr4_worker_test_module", "_CR4PrefetchCapability")
    with _TestClient(app) as client:
        resp = client.post("/prefetch")
        assert resp.status_code == 200, f"expected 200, got {resp.status_code}: {resp.text}"
        assert resp.json()["status"] == "prefetched"
        assert _CR4PrefetchCapability.prefetch_calls == 1, "prefetch hook must fire"
        
        # /reconfigure success path with the request body shape the helper expects
        resp = client.post("/reconfigure", json={
            "old_config": {"model": "base"},
            "new_config": {"model": "large"},
        })
        assert resp.status_code == 200, f"expected 200, got {resp.status_code}: {resp.text}"
        assert resp.json()["status"] == "reconfigured"
        assert _CR4PrefetchCapability.reconfigure_calls == [
            ({"model": "base"}, {"model": "large"})
        ], f"reconfigure args wrong: {_CR4PrefetchCapability.reconfigure_calls}"
    
    # /prefetch raising path → 500 with detail (NOT 200 + status=error;
    # mirrors /execute's "raised → 500" semantics)
    app_raising = create_app("_cr4_worker_test_module", "_CR4PrefetchRaisingCapability")
    with _TestClient(app_raising) as client:
        resp = client.post("/prefetch")
        assert resp.status_code == 500, f"expected 500 on raising prefetch, got {resp.status_code}"
        assert "prefetch boom" in resp.json()["detail"]
    
    # /execute → 409 on CapabilityCancelledError (cancellation distinct from 500)
    app_cancel = create_app("_cr4_worker_test_module", "_CR4CancellingCapability")
    with _TestClient(app_cancel) as client:
        resp = client.post("/execute", json={"args": [], "kwargs": {}})
        assert resp.status_code == 409, \
            f"CapabilityCancelledError must surface as 409 not {resp.status_code}: {resp.text}"
        assert "cr4-cancelling-capability" in resp.json()["detail"]
    
    # /execute → 500 on real failure (proves the 409 path is specifically for
    # CapabilityCancelledError, not all exceptions)
    app_fail = create_app("_cr4_worker_test_module", "_CR4FailingCapability")
    with _TestClient(app_fail) as client:
        resp = client.post("/execute", json={"args": [], "kwargs": {}})
        assert resp.status_code == 500, f"real failure must remain 500, got {resp.status_code}"
        # G7: the 500 body now carries the typed _job_error sentinel (SG-52
    # parity for the unary path) instead of a bare-string detail.
    body = resp.json()
    assert "_job_error" in body, f"expected _job_error sentinel: {body}"
    assert "simulated capability failure" in body["_job_error"]["message"]
    assert body["_job_error"].get("category"), "JobError category must survive the wire"


_cr4_worker_endpoints_check()
print("CR-4 worker endpoints (/prefetch, /reconfigure, 409 cancel): PASS")

In [ ]:
#| hide
# SG-51 + SG-52 worker tests: /execute_stream resets _cancel_requested and emits
# typed JobError chunks via the `_job_error` sentinel key (replacing the
# undocumented {"error": str(e)} shape).

_fake_mod_sg51 = _types_cr3.ModuleType("_sg51_sg52_worker_test_module")


class _SG51StreamCapability:
    """Capability whose execute_stream yields items normally. Used to verify SG-51's
    flag reset doesn't break the normal success path."""
    name = "sg51-stream-capability"
    version = "1.0.0"
    _cancel_requested = True  # Pre-set to True: SG-51 reset must clear before iteration
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self): return {}
    def get_current_config(self): return {}
    def execute_stream(self, *args, **kwargs):
        # If SG-51's reset works, this yields all three chunks. If the leftover
        # _cancel_requested=True propagated, a real capability would check_cancel()
        # and raise mid-stream — but here we just verify the iteration sees the
        # flag reset by reading it after we've started yielding.
        yield {"chunk": 0, "cancel_flag_at_start": self._cancel_requested}
        yield {"chunk": 1}
        yield {"chunk": 2}


class _SG52CancellingStreamCapability:
    """Capability whose execute_stream raises CapabilityCancelledError mid-stream — verifies
    SG-52's typed error chunk emission and category mapping for cancellation."""
    name = "sg52-cancel-capability"
    version = "1.0.0"
    _cancel_requested = False
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self): return {}
    def get_current_config(self): return {}
    def execute_stream(self, *args, **kwargs):
        yield {"data": "first"}
        # Mid-stream cancellation: in production this happens via check_cancel()
        # after the substrate flipped _cancel_requested via /cancel.
        raise _CapabilityCancelledError_CR4(self.name)


class _SG52FailingStreamCapability:
    """Capability whose execute_stream raises a bare RuntimeError — verifies SG-52's
    typed error chunk uses CR-5's default classification (RuntimeError → fatal)."""
    name = "sg52-fail-capability"
    version = "1.0.0"
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self): return {}
    def get_current_config(self): return {}
    def execute_stream(self, *args, **kwargs):
        yield {"data": "before failure"}
        raise RuntimeError("simulated stream failure")


_fake_mod_sg51._SG51StreamCapability = _SG51StreamCapability
_fake_mod_sg51._SG52CancellingStreamCapability = _SG52CancellingStreamCapability
_fake_mod_sg51._SG52FailingStreamCapability = _SG52FailingStreamCapability
_sys_cr3.modules["_sg51_sg52_worker_test_module"] = _fake_mod_sg51


def _read_ndjson_chunks(response) -> list:
    """Helper: read NDJSON chunks from a streaming response."""
    chunks = []
    for line in response.iter_lines():
        if line:
            chunks.append(json.loads(line))
    return chunks


def _sg51_cancel_flag_reset_check():
    """SG-51: /execute_stream resets _cancel_requested before iter_response runs."""
    app = create_app("_sg51_sg52_worker_test_module", "_SG51StreamCapability")
    with _TestClient(app) as client:
        with client.stream("POST", "/execute_stream", json={"args": [], "kwargs": {}}) as resp:
            assert resp.status_code == 200
            chunks = _read_ndjson_chunks(resp)
        # SG-51 invariant: the leftover _cancel_requested=True on the class must
        # be cleared by the worker before yielding the first chunk. The capability
        # captures the flag's value at yield-time in the first chunk's payload.
        assert chunks[0]["cancel_flag_at_start"] is False, \
            f"SG-51: worker did NOT reset _cancel_requested; got {chunks[0]}"
        assert chunks == [
            {"chunk": 0, "cancel_flag_at_start": False},
            {"chunk": 1},
            {"chunk": 2},
        ], f"unexpected chunks: {chunks}"


def _sg52_typed_cancel_chunk_check():
    """SG-52: CapabilityCancelledError mid-stream emits a typed _job_error chunk
    with category=transient + original_exc_repr identifying the cancellation."""
    app = create_app("_sg51_sg52_worker_test_module", "_SG52CancellingStreamCapability")
    with _TestClient(app) as client:
        with client.stream("POST", "/execute_stream", json={"args": [], "kwargs": {}}) as resp:
            assert resp.status_code == 200, "stream errors are emitted in-band, not via status code"
            chunks = _read_ndjson_chunks(resp)
        # First chunk is normal capability output
        assert chunks[0] == {"data": "first"}
        # Terminal chunk is the typed _job_error sentinel
        assert len(chunks) == 2, f"expected exactly 2 chunks, got {len(chunks)}: {chunks}"
        terminal = chunks[1]
        assert "_job_error" in terminal, f"missing _job_error sentinel: {terminal}"
        je = terminal["_job_error"]
        # Category-based dispatch info
        assert je["category"] == "transient", f"CapabilityCancelledError → transient: got {je['category']}"
        # CapabilityCancelledError-specific recognition signal (used by proxy)
        assert je["original_exc_repr"].startswith("CapabilityCancelledError"), \
            f"original_exc_repr must identify cancellation: {je['original_exc_repr']}"
        # default_retriable=False on CapabilityCancelledError flows through to JobError
        assert je["retriable"] is False, "CapabilityCancelledError must NOT be auto-retriable"
        assert je["capability_name"] == "sg52-cancel-capability"
        # JobError.occurred_at serialized via the EnhancedJSONEncoder datetime path
        assert je["occurred_at"] is not None, "occurred_at must serialize"


def _sg52_typed_failure_chunk_check():
    """SG-52: bare RuntimeError mid-stream emits a typed _job_error chunk with
    CR-5 default classification (RuntimeError → fatal, retriable=False)."""
    app = create_app("_sg51_sg52_worker_test_module", "_SG52FailingStreamCapability")
    with _TestClient(app) as client:
        with client.stream("POST", "/execute_stream", json={"args": [], "kwargs": {}}) as resp:
            assert resp.status_code == 200
            chunks = _read_ndjson_chunks(resp)
        assert chunks[0] == {"data": "before failure"}
        terminal = chunks[1]
        assert "_job_error" in terminal
        je = terminal["_job_error"]
        assert je["category"] == "fatal", f"bare RuntimeError → fatal: got {je['category']}"
        assert je["retriable"] is False
        assert "simulated stream failure" in je["message"]
        assert je["capability_name"] == "sg52-fail-capability"


_sg51_cancel_flag_reset_check()
_sg52_typed_cancel_chunk_check()
_sg52_typed_failure_chunk_check()
print("SG-51 + SG-52 /execute_stream cancel-reset + typed JobError chunks: PASS")

In [ ]:
#| hide
# CR-14 worker test: the per-call envelope reaches capability code in the executor
# thread (the run_in_executor + copy_context path) — and an envelope-less call
# leaves the contextvar honestly None.
from cjm_substrate.core.wire import get_call_envelope as _get_env_cr14

_fake_mod_cr14 = _types_cr3.ModuleType("_cr14_worker_test_module")


class _CR14EnvelopeCapability:
    """Stub capability that captures the contextvar AS SEEN INSIDE execute()."""
    name = "cr14-envelope-capability"
    version = "1.0.0"
    seen_envelopes = []  # class-level capture across calls
    def initialize(self, config=None): pass
    def execute(self, *args, **kwargs):
        env = _get_env_cr14()
        type(self).seen_envelopes.append(env)
        return {"ok": True}
    def get_config_schema(self): return {}
    def get_current_config(self): return {}


_fake_mod_cr14._CR14EnvelopeCapability = _CR14EnvelopeCapability
_sys_cr3.modules["_cr14_worker_test_module"] = _fake_mod_cr14


def _cr14_envelope_check():
    _CR14EnvelopeCapability.seen_envelopes = []
    app = create_app("_cr14_worker_test_module", "_CR14EnvelopeCapability")
    with _TestClient(app) as client:
        # (a) Envelope present: capability code sees exact identity in the
        # executor thread (copy_context carried it across).
        resp = client.post("/execute", json={
            "args": [], "kwargs": {"x": 1},
            "envelope": {"job_id": "job-cr14", "run_id": "run-7",
                         "control": {"force": True},
                         "future_key_ignored": "tolerant"},
        })
        assert resp.status_code == 200, resp.text
        # (b) Envelope absent: contextvar is None (honest unattribution),
        # never a stale carry-over from the previous request.
        resp = client.post("/execute", json={"args": [], "kwargs": {}})
        assert resp.status_code == 200, resp.text
    seen = _CR14EnvelopeCapability.seen_envelopes
    assert len(seen) == 2, seen
    assert seen[0] is not None and seen[0].job_id == "job-cr14"
    assert seen[0].run_id == "run-7" and seen[0].control == {"force": True}
    assert seen[1] is None, "envelope must NOT leak across requests"


_cr14_envelope_check()
print("CR-14 worker call-envelope propagation: PASS")

In [ ]:
#| hide
# CR-14 follow-up worker tests: in-worker accounts ride the X-CJM-Accounts
# response header on the unary paths — success AND the _job_error 500; the
# /task endpoint records TASK_ACCOUNT itself (zero capability cooperation);
# account-less calls carry NO header (byte-identical responses for old hosts).
from cjm_substrate.core.wire import (
    ACCOUNTS_HEADER as _AH_acct, record_account as _ra_acct,
)

_fake_mod_acct = _types_cr3.ModuleType("_acct_worker_test_module")


class _AcctRecordingCapability:
    """Capability whose execute records accounts (the T29 storage-helper shape)."""
    name = "acct-recording-capability"
    version = "1.0.0"
    def initialize(self, config=None): pass
    def execute(self, *args, mode="save", **kwargs):
        if mode == "save":
            _ra_acct("result_saved", {"row_job_id": "row-1",
                                      "text_hash": "sha256:abc"})
            return {"ok": True}
        if mode == "save_then_fail":
            _ra_acct("result_saved", {"row_job_id": "row-2"})
            raise RuntimeError("post-save crash")
        return {"ok": True}  # mode="plain": no accounts recorded
    def get_config_schema(self): return {}
    def get_current_config(self): return {}


class _AcctTaskAdapter:
    """Minimal task adapter: /task dispatch target for the TASK_ACCOUNT test."""
    task_name = "acct-task"
    def __init__(self, tool): self.tool = tool
    def do_thing(self, x=0):
        if x < 0:
            raise RuntimeError("negative")
        return {"x": x}


_fake_mod_acct._AcctRecordingCapability = _AcctRecordingCapability
_fake_mod_acct._AcctTaskAdapter = _AcctTaskAdapter
_sys_cr3.modules["_acct_worker_test_module"] = _fake_mod_acct


def _acct_worker_check():
    app = create_app("_acct_worker_test_module", "_AcctRecordingCapability",
                     adapter_specs=["_acct_worker_test_module:_AcctTaskAdapter"])
    with _TestClient(app) as client:
        # (a) Success path: recorded account arrives on the header.
        resp = client.post("/execute", json={"args": [], "kwargs": {"mode": "save"}})
        assert resp.status_code == 200
        accounts = json.loads(resp.headers[_AH_acct])
        assert accounts == [{"event_type": "result_saved",
                             "payload": {"row_job_id": "row-1",
                                         "text_hash": "sha256:abc"}}]

        # (b) No accounts recorded -> header ABSENT (old-host byte parity).
        resp = client.post("/execute", json={"args": [], "kwargs": {"mode": "plain"}})
        assert resp.status_code == 200
        assert _AH_acct not in resp.headers

        # (c) Failure path: the save that happened BEFORE the crash still
        # reports on the 500's header beside the _job_error body.
        resp = client.post("/execute", json={"args": [], "kwargs": {"mode": "save_then_fail"}})
        assert resp.status_code == 500
        assert "_job_error" in resp.json()
        accounts = json.loads(resp.headers[_AH_acct])
        assert accounts[0]["payload"]["row_job_id"] == "row-2"

        # (d) /task: worker-emitted TASK_ACCOUNT, ok=True + duration.
        resp = client.post("/task", json={"task": "acct-task", "method": "do_thing",
                                          "kwargs": {"x": 5}})
        assert resp.status_code == 200 and resp.json() == {"x": 5}
        accounts = json.loads(resp.headers[_AH_acct])
        assert accounts[0]["event_type"] == "task_account"
        assert accounts[0]["payload"]["ok"] is True
        assert accounts[0]["payload"]["task"] == "acct-task"
        assert "duration_s" in accounts[0]["payload"]

        # (e) /task failure: TASK_ACCOUNT with ok=False + error category.
        resp = client.post("/task", json={"task": "acct-task", "method": "do_thing",
                                          "kwargs": {"x": -1}})
        assert resp.status_code == 500
        accounts = json.loads(resp.headers[_AH_acct])
        assert accounts[0]["payload"]["ok"] is False
        assert accounts[0]["payload"]["error_category"] == "fatal"

        # (f) No cross-request leakage: a plain call after account-recording
        # calls still carries no header.
        resp = client.post("/execute", json={"args": [], "kwargs": {"mode": "plain"}})
        assert _AH_acct not in resp.headers


_acct_worker_check()
print("CR-14 follow-up worker accounts header: PASS")

## CLI Entry Point

The worker is launched as a subprocess with the capability module/class and port specified via command line arguments.

In [ ]:
#| export
def run_worker() -> None:
    """CLI entry point for running the worker."""
    parser = argparse.ArgumentParser(description="Universal Capability Worker")
    parser.add_argument("--module", required=True, help="Capability module path")
    parser.add_argument("--class", dest="class_name", required=True, help="Capability class name")
    parser.add_argument("--adapters", required=False, default="",
                        help="Comma-separated adapter impl specs 'module:ClassName' (CR-17 pt 2)")
    # SG-4: parent-bound listening-socket FD inheritance closes the
    # bind-then-close-then-reopen TOCTOU race. --fd is preferred on Unix;
    # --port stays as the Windows fallback (pass_fds is Unix-only) and as a
    # diagnostic escape hatch when invoking the worker directly.
    parser.add_argument("--fd", type=int, required=False, default=None,
                        help="Inherited listening-socket file descriptor (Unix; preferred)")
    parser.add_argument("--port", type=int, required=False, default=None,
                        help="Port to bind (used when --fd is not provided)")
    parser.add_argument("--ppid", type=int, required=False, help="Parent PID to monitor")
    args = parser.parse_args()

    if args.fd is None and args.port is None:
        parser.error("one of --fd or --port is required")

    # Start watchdog if parent PID provided
    if args.ppid:
        watchdog = threading.Thread(
            target=parent_monitor,
            args=(args.ppid,),
            daemon=True
        )
        watchdog.start()

    adapter_specs = [s.strip() for s in (args.adapters or "").split(",") if s.strip()]
    app = create_app(args.module, args.class_name, adapter_specs=adapter_specs)
    if args.fd is not None:
        uvicorn.run(app, fd=args.fd, log_level="warning")
    else:
        uvicorn.run(app, host="127.0.0.1", port=args.port, log_level="warning")

### Usage

The worker is typically launched by the `RemoteCapabilityProxy`:

```bash
# Example: Launch a Whisper capability worker
python -m cjm_substrate.core.worker \
    --module cjm_transcription_capability_whisper.capability \
    --class WhisperLocalCapability \
    --port 12345 \
    --ppid 1234
```

The `--ppid` argument enables the suicide pact: if process 1234 dies, this worker terminates.

In [ ]:
#| export
#| eval: false
import sys

if __name__ == "__main__" and "ipykernel" not in sys.modules:
    run_worker()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()